***Import packages***

In [13]:
!pip install opencv-python

In [14]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import re
tqdm_notebook.pandas()
import tensorflow as tf
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Flatten, GlobalAveragePooling2D, Conv2D, MaxPooling2D
from tensorflow.keras import callbacks
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.efficientnet import EfficientNetB7
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications.densenet import DenseNet121
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator, image
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score, recall_score, accuracy_score, precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn import metrics
import itertools
import pickle
from joblib import dump, load


ModuleNotFoundError: No module named 'tensorflow.keras'

# Load cleaned DF

In [24]:
# Importer la Dataframe
df =  pd.read_csv('df_train_final.csv')

df = df.drop(["Unnamed: 0","designation","description","productid","imageid"], axis=1)

# Fonction pour extraire le nom de fichier
def extraire_nom_img(row):
    chemin_complet = row['nom_img']
    nom_fichier = chemin_complet.split('\\')[-1]  # Diviser le chemin complet et extraire le dernier élément (le nom du fichier)
    return nom_fichier

# Appliquer la fonction à la colonne 'nom_img' pour créer la nouvelle colonne 'nom_img_nette'
df['nom_img_nette'] = df.apply(lambda row: extraire_nom_img(row), axis=1)


# L'emplacement que vous souhaitez ajouter
emplacement = 'C:\\Users\\yahya\\OneDrive\Documents\\Rakuten\\images\\images\\image_resized\\'

# Fonction pour ajouter l'emplacement au nom de l'image
def ajouter_emplacement(nom_img):
    return emplacement + nom_img + ".jpg"

# Appliquer la fonction à la colonne 'nom_img'
df['nom_img'] = df['nom_img'].apply(ajouter_emplacement)

df.head()
 

# Importer les modèles pré-entrainé

# Load cleaned DF

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# Custom function to load the model without compiling it
def load_model_custom(chemin):
    return load_model(chemin, compile=False)

# Load Pre-trained Model Simple DNN 
simpleDnnNet = load_model_custom("checkpoint_SimpleDNN_09032024_LR_0.001_100Epochs.h5")

# Load Pre-trained Model Conv1D
conv1DNet = load_model_custom("CONV1Dcheckpoint_Conv1D_09032024_LR_0.001_50Epochs.h5")


# Load Pre-trained Model XCeption
XceptionV3 = load_model_custom("checkpoint_Xception__Shape_250x250_25Epochs.h5.keras")

# Load pre-trained models

# Séparation des variables

In [27]:
X_train, X_test = train_test_split(df, test_size=0.2)

In [28]:
#X_test['nom_img_+_dir'] = X_test['nom_img']
y_test_true = X_test['cat_code']

In [ ]:
X_test

In [30]:
def tokenize_text(X_train_text, X_test_text):
  
    #maxlen = 500
    maxlen = 400
    num_words = 20000
    
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words = num_words)
    tokenizer.fit_on_texts(X_train_text)
    
    X_train = tokenizer.texts_to_sequences(X_train_text)
    X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train,
                                                            maxlen = maxlen,
                                                            padding='post')
    X_test = tokenizer.texts_to_sequences(X_test_text)
    
    X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test,
                                                           maxlen = maxlen,
                                                           padding='post')
    print("X_train text created. Size : ", X_train.shape)
    print("X_test text created. Size : ", X_test.shape)    
    
    return X_train, X_test 

In [31]:
def tokenize_text_test_only(X_test_text):
  
    #maxlen = 500
    maxlen = 400
    num_words = 20000
    
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words = num_words)
    tokenizer.fit_on_texts(X_test_text)
    
    X_test = tokenizer.texts_to_sequences(X_train_text)
    X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test,
                                                            maxlen = maxlen,
                                                            padding='post')

    print("X_test text created. Size : ", X_test.shape)    
    
    return X_train, X_test 

In [32]:
def predict_images(model,im_shape, fullpath):
    
    #fct_name = '[INFO]-[predict_images_proba]'

    y_pred_proba_list = []
    
    for filepath in tqdm(nom_img):
        im = cv2.resize(cv2.imread(filepath), im_shape)
        im = im/255
        im = np.expand_dims(im, axis = 0)
        
        out = model.predict(im)
        
        y_pred_proba_list.append(out)
        
      
    y_pred_proba = np.array(y_pred_proba_list)
    y_pred_proba = y_pred_proba.reshape(y_pred_proba.shape[0],27)
    
    permutation = [0, 1, 12, 20, 21, 22, 23, 24, 25, 26, 2, 3,
                   4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16
                   ,17, 18, 19]
    # swap columns matchs cnn label with real class number
    y_pred_proba = y_pred_proba[:, permutation] 
    y_pred_class = np.argmax(y_pred_proba,axis = 1).astype(int)  # pred with real class labels
    
    #display(fct_name + 'Probabilities real class num shape:' ,y_pred_proba.shape)
    #display(fct_name + 'Predicted real class labels(0 to 26) shape:',y_pred_class.shape)
    
    return y_pred_proba,  y_pred_class

In [33]:
image_Dir= "C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\images\\images\\image_resized\\"

def predict_images_data_gen(model, test_df, input_shape, image_Dir):    
 
        test_df.prdtypecode =   test_df.prdtypecode.astype(str)


        
        datagen = ImageDataGenerator(
                                    rescale=1./255
                                    )
        
           
        test_img_gen = datagen.flow_from_dataframe(dataframe = test_df, 
                                                    directory = image_Dir,
                                                    x_col='nom_img_nette', 
                                                    y_col='cat_code',                                                                                 
                                                    class_mode='sparse',
                                                    target_size=(input_shape[0], input_shape[1]),
                                                    batch_size= 64, 
                                                    shuffle = False                                                   
                                                   )
        
        y_pred_proba = get_true_class(model, test_img_gen)
        return y_pred_proba

def get_true_class(model, data_gen):
    
    data_gen.reset()
    
    #test_score_cnn = model.evaluate(data_gen)
    
    pred = model.predict(data_gen, verbose = 1)
  
    class_dict =  data_gen.class_indices  # a dictionary where key is the class name (  assigned by Image Data Generator)
                                           #and value is the corresponding label of real class of dataset of images
        
   # List of permutation corresponding to real class number
   
    permutation = [data_gen.class_indices[str(i)] for i in range(len(data_gen.class_indices))]
    
    #print(permutation)   
    
    y_pred_proba = pred[:, permutation] # swap columns
    pred_class_cnn = np.argmax(y_pred_proba, axis = 1).astype(int)
    
    return y_pred_proba, pred_class_cnn

# Predict with Images - Xception

## Predict Images One by One

y_pred_proba_cnn_1, pred_class_cnn_1 = predict_images(XceptionV3,
                                                  (250, 250),
                                                  X_test['nom_img'])

In [35]:
y_pred_proba_cnn_2, pred_class_cnn_2 = predict_images_data_gen(XceptionV3,
                                                           X_test,                                                          
                                                           (250, 250),
                                                           image_Dir)

Found 16984 validated image filenames belonging to 27 classes.
266/266 [==============================] - 77s 282ms/step


## Final CNN prediction

***Final CNN prediction***

In [38]:
y_pred_proba_cnn = y_pred_proba_cnn_1
pred_class_cnn =  pred_class_cnn_1

In [ ]:
y_pred_proba_cnn = y_pred_proba_cnn_2
pred_class_cnn =  pred_class_cnn_2

In [39]:
# avec le num de classe reel
print(metrics.classification_report(y_test_true , pred_class_cnn))

              precision    recall  f1-score   support

           0       0.58      0.65      0.62       643
           1       0.68      0.59      0.63       503
           2       0.46      0.48      0.47       319
           3       0.79      0.73      0.76       179
           4       0.71      0.66      0.68       545
           5       0.92      0.92      0.92       793
           6       0.55      0.31      0.40       144
           7       0.55      0.41      0.47      1002
           8       0.33      0.36      0.34       408
           9       0.71      0.79      0.75       993
          10       0.67      0.57      0.61       162
          11       0.41      0.61      0.49       514
          12       0.53      0.51      0.52       664
          13       0.53      0.63      0.58       993
          14       0.81      0.84      0.83       832
          15       0.77      0.64      0.70       159
          16       0.51      0.59      0.55       970
          17       0.69    

# Predict withText - Simple DNN & Embedding + Conv1D

In [40]:
X_train_text, X_test_text = tokenize_text(X_train['Texte'], X_test['Texte'])

X_train text created. Size :  (67932, 400)
X_test text created. Size :  (16984, 400)


# Simple DNN

In [41]:
y_pred_proba_rnn_1 = simpleDnnNet.predict(X_test_text)
y_pred_rnn_1_class = np.argmax(y_pred_proba_rnn_1,axis = 1).astype(int)

In [42]:
# Report Text predictions
print(metrics.classification_report(y_test_true, y_pred_rnn_1_class))

              precision    recall  f1-score   support

           0       0.40      0.62      0.49       643
           1       0.75      0.62      0.68       503
           2       0.77      0.86      0.81       319
           3       0.97      0.82      0.89       179
           4       0.77      0.80      0.79       545
           5       0.87      0.90      0.88       793
           6       0.75      0.47      0.58       144
           7       0.72      0.73      0.73      1002
           8       0.68      0.54      0.60       408
           9       0.94      0.94      0.94       993
          10       0.97      0.88      0.92       162
          11       0.84      0.77      0.80       514
          12       0.79      0.78      0.79       664
          13       0.80      0.86      0.83       993
          14       0.86      0.95      0.90       832
          15       0.85      0.80      0.82       159
          16       0.83      0.78      0.80       970
          17       0.93    

# Embedding + Conv1D

***Embedding + Conv1D***

In [43]:
#CONV1D
y_pred_proba_rnn_2 = conv1DNet.predict(X_test_text)
y_pred_rnn_2_class = np.argmax(y_pred_proba_rnn_2,axis = 1).astype(int)

In [44]:
# Report Text predictions
print(metrics.classification_report(y_test_true, y_pred_rnn_2_class))

              precision    recall  f1-score   support

           0       0.40      0.56      0.47       643
           1       0.71      0.65      0.68       503
           2       0.83      0.76      0.79       319
           3       0.88      0.84      0.86       179
           4       0.73      0.81      0.77       545
           5       0.84      0.89      0.86       793
           6       0.89      0.50      0.64       144
           7       0.70      0.74      0.72      1002
           8       0.63      0.51      0.56       408
           9       0.95      0.93      0.94       993
          10       0.99      0.89      0.94       162
          11       0.80      0.76      0.78       514
          12       0.82      0.75      0.79       664
          13       0.85      0.81      0.83       993
          14       0.90      0.96      0.92       832
          15       0.84      0.81      0.82       159
          16       0.79      0.82      0.81       970
          17       0.84    

# Models Voting

## Max Voting

In [45]:
final_pred_max_proba =np.maximum(y_pred_proba_cnn, y_pred_proba_rnn_1, y_pred_proba_rnn_2)

In [46]:
final_pred_max = np.argmax(final_pred_max_proba,axis = 1).astype(int)

In [48]:
print(metrics.classification_report(y_test_true, final_pred_max))

              precision    recall  f1-score   support

           0       0.71      0.67      0.69       643
           1       0.85      0.72      0.78       503
           2       0.79      0.80      0.79       319
           3       0.91      0.83      0.87       179
           4       0.80      0.83      0.82       545
           5       0.93      0.99      0.96       793
           6       0.78      0.47      0.58       144
           7       0.75      0.69      0.72      1002
           8       0.58      0.53      0.55       408
           9       0.90      0.95      0.93       993
          10       0.90      0.88      0.89       162
          11       0.74      0.79      0.76       514
          12       0.82      0.75      0.78       664
          13       0.79      0.85      0.82       993
          14       0.86      0.95      0.90       832
          15       0.87      0.82      0.84       159
          16       0.78      0.78      0.78       970
          17       0.95    

***Weighted Average***

# Weighted Average

In [50]:
xception_score = 0.66
conv1D_score = 0.80
simplDNN_score = 0.81

final_pred_weighted_proba = (y_pred_proba_cnn * (xception_score) +    
                            y_pred_proba_rnn_1 * (simplDNN_score) +  
                            y_pred_proba_rnn_2 * (conv1D_score))/(xception_score + simplDNN_score + conv1D_score)


In [51]:
final_pred_weighted = np.argmax(final_pred_weighted_proba,axis = 1).astype(int)

In [53]:
print(metrics.classification_report(y_test_true, final_pred_weighted))

              precision    recall  f1-score   support

           0       0.69      0.70      0.69       643
           1       0.85      0.74      0.79       503
           2       0.79      0.82      0.81       319
           3       0.92      0.83      0.87       179
           4       0.81      0.83      0.82       545
           5       0.93      0.99      0.96       793
           6       0.81      0.47      0.59       144
           7       0.75      0.73      0.74      1002
           8       0.62      0.54      0.58       408
           9       0.92      0.95      0.93       993
          10       0.92      0.88      0.90       162
          11       0.76      0.80      0.78       514
          12       0.84      0.78      0.81       664
          13       0.80      0.87      0.83       993
          14       0.87      0.95      0.91       832
          15       0.87      0.84      0.85       159
          16       0.79      0.80      0.80       970
          17       0.96    

***Save results***